## Add `med_type` to an asthma NDC spreadsheet

This notebook adds a **`med_type`** column for high-level grouping (**Rescue**, **Controller**, **Montelukast**) from the existing detailed class column **`asthma_med_class_comp`**. Values that do not match any list stay missing (`<NA>`).

### Files

- **Input**: `INPUT_XLSX` (Excel) — must include column `asthma_med_class_comp`.
- **Output**: `OUTPUT_XLSX` (Excel) — same rows/columns as input, plus `med_type`.

Edit `INPUT_XLSX` / `OUTPUT_XLSX` in the code cell to match your filenames.

### Mapping (summary)

- **Rescue** — e.g. SABA inhaler (inhaled SABA; plain `SABA` in `asthma_med_class_comp` is rewritten to this label), SABA oral/IV, SAMA, SABA combinations, epinephrine inhaler, OCS, IV methylxanthine (see sets in the code cell).
- **Controller** — e.g. ICS/LABA/LAMA and combinations, oral methylxanthine, mast cell stabilizers, non-montelukast LTRA, 5-LOX, biologic (labels must match `asthma_med_class_comp` after trimming surrounding whitespace).
- **Montelukast** — class value `Montelukast` only (same trimming rule).

Rows with `asthma_med_class_comp` not listed above (or blank) get `<NA>` in `med_type`. After running, use the printed `value_counts` to sanity-check how many rows were labeled.

### Related documentation

See **`README.md`** and **`asthma_medication_approach.pdf`** for the broader medication-classification context.


In [3]:
from pathlib import Path

import pandas as pd

INPUT_XLSX = Path("Asthma NDCs_5.9.2026.xlsx")
OUTPUT_XLSX = Path("Asthma NDCs_5.9.2026_added.xlsx")

rescue_meds = {
    "SABA inhaler",
    "SABA_oral",
    "SABA_iv",
    "Epinephrine_inhaler",
    "SAMA",
    "SABA/SAMA",
    "SABA/ICS",
    "Methylxanthine_iv",
}
controller_meds = {
    "ICS",
    "LABA",
    "LAMA",
    "ICS/LABA",
    "LABA/LAMA",
    "ICS/LABA/LAMA",
    "Methylxanthine_oral",
    "Mast_cell_stabilizers",
    "Non_Montelukast_LTRA",
    "5_LOX",
    "Biologic",
}
montelukast_meds = {"Montelukast"}

data = pd.read_excel(INPUT_XLSX, dtype={"NDC": "string"})
data["NDC"] = data["NDC"].astype("string").str.strip()
# Trim surrounding whitespace so values like "Montelukast " match "Montelukast".
col = data["asthma_med_class_comp"].astype("string").str.strip()
# In this list, plain "SABA" means inhaled SABA (same as "SABA inhaler").
col = col.replace("SABA", "SABA inhaler")
data["asthma_med_class_comp"] = col
# Same order as R case_when (first match wins). Unmatched rows stay NA.
med_type = pd.Series(pd.NA, index=data.index, dtype="string")
med_type.loc[col.isin(rescue_meds)] = "Rescue"
med_type.loc[col.isin(controller_meds) & med_type.isna()] = "Controller"
med_type.loc[col.isin(montelukast_meds) & med_type.isna()] = "Montelukast"
data["med_type"] = med_type

data.to_excel(OUTPUT_XLSX, index=False)
print(f"Wrote {OUTPUT_XLSX.resolve()} ({len(data)} rows).")
print(data["med_type"].value_counts(dropna=False))

Wrote C:\Users\shiroa1\OneDrive - VUMC\inhaler_helper_functions\Asthma NDCs_5.9.2026_added.xlsx (23469 rows).
med_type
<NA>           20085
Controller      1442
Rescue          1433
Montelukast      509
Name: count, dtype: Int64
